# 🎙 My Voice — train your own voice model

Turns the recordings you made in the **My Voice** app into a voice model that sounds like you and runs offline on your phone.

**Before you start**
1. In the app: **Train** tab → record the sentences → **Export recordings**. You get `myvoice-dataset.zip` in *Downloads/MyVoice*.
2. Upload that zip to Google Drive into a folder named **MyVoice** (so it's at `My Drive/MyVoice/myvoice-dataset.zip`).
3. Here: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Training takes roughly 1–3 hours for 1000 epochs. Progress is saved to Drive, so if Colab disconnects just press **Run all** again and it continues.
When it finishes, `my_voice.onnx` and `my_voice.onnx.json` appear in `My Drive/MyVoice` — import both in the app (**Train** tab → *Import my model*).

In [ ]:
#@title 1 · Settings
DATASET_ZIP = "/content/drive/MyDrive/MyVoice/myvoice-dataset.zip"  #@param {type:"string"}
VOICE_NAME = "my_voice"  #@param {type:"string"}
EPOCHS = 1000  #@param {type:"integer"}
#@markdown Starting voice to adapt from. `ljspeech` has public-domain training data (safe for monetised videos). `lessac` is a popular base but its data is non-commercial.
BASE_VOICE = "ljspeech"  #@param ["ljspeech", "ryan", "lessac", "kusal"]
BATCH_SIZE = 16  #@param {type:"integer"}
#@markdown Tick to ignore a previous unfinished run and start over (e.g. after re-recording).
START_OVER = False  #@param {type:"boolean"}


In [ ]:
#@title 2 · Connect Google Drive and check the GPU
import subprocess, os
from google.colab import drive
drive.mount('/content/drive')
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print(gpu or '⚠️ No GPU! Runtime → Change runtime type → T4 GPU, then run again.')
WORK = '/content/drive/MyDrive/MyVoice'
os.makedirs(WORK, exist_ok=True)
assert os.path.exists(DATASET_ZIP), f'Could not find {DATASET_ZIP}. Upload myvoice-dataset.zip to My Drive/MyVoice first.'
print('Dataset found ✓')


In [ ]:
#@title 3 · Install the training tools (~3 min)
!pip -q install "piper-tts[train]==1.8.0" onnxscript huggingface_hub cython 2>&1 | grep -v -i "warning\|already" | tail -3
import os, sys, subprocess, piper, pathlib
ma = pathlib.Path(piper.__file__).parent / 'train/vits/monotonic_align'
(ma / 'monotonic_align').mkdir(exist_ok=True)
(ma / 'monotonic_align/__init__.py').touch()
os.makedirs('/content/ma', exist_ok=True)
open('/content/ma/core.pyx', 'w').write('''cimport cython
from cython.parallel import prange

@cython.boundscheck(False)
@cython.wraparound(False)
cdef void maximum_path_each(int[:,::1] path, float[:,::1] value, int t_y, int t_x, float max_neg_val=-1e9) nogil:
  cdef int x
  cdef int y
  cdef float v_prev
  cdef float v_cur
  cdef int index = t_x - 1
  for y in range(t_y):
    for x in range(max(0, t_x + y - t_y), min(t_x, y + 1)):
      if x == y:
        v_cur = max_neg_val
      else:
        v_cur = value[y-1, x]
      if x == 0:
        if y == 0:
          v_prev = 0.
        else:
          v_prev = max_neg_val
      else:
        v_prev = value[y-1, x-1]
      value[y, x] += max(v_prev, v_cur)
  for y in range(t_y - 1, -1, -1):
    path[y, index] = 1
    if index != 0 and (index == y or value[y-1, index] < value[y-1, index-1]):
      index = index - 1

@cython.boundscheck(False)
@cython.wraparound(False)
cpdef void maximum_path_c(int[:,:,::1] paths, float[:,:,::1] values, int[::1] t_ys, int[::1] t_xs) nogil:
  cdef int b = paths.shape[0]
  cdef int i
  for i in prange(b, nogil=True):
    maximum_path_each(paths[i], values[i], t_ys[i], t_xs[i])
''')
open('/content/ma/setup.py', 'w').write('from setuptools import setup\nfrom Cython.Build import cythonize\nimport numpy\nsetup(ext_modules=cythonize("core.pyx"), include_dirs=[numpy.get_include()])\n')
subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace', '-q'], cwd='/content/ma', check=True, capture_output=True)
import glob, shutil
for so in glob.glob('/content/ma/core*.so'):
    shutil.copy(so, ma / 'monotonic_align')
from piper.train.vits.monotonic_align import maximum_path
print('Training tools ready ✓')


In [ ]:
%%writefile /content/train_voice.py
"""Fine-tune a Piper voice on your recordings.

Same as `python -m piper.train fit ...` but with checkpointing tuned for Colab:
no 846 MB checkpoint per epoch; instead a resume copy every BACKUP_EVERY epochs
(to Google Drive) and a final checkpoint when training ends.
"""
import os
from pathlib import Path

from lightning.pytorch.callbacks import Callback
from piper.train.__main__ import VitsLightningCLI
from piper.train.vits.dataset import VitsDataModule
from piper.train.vits.lightning import VitsModel

BACKUP = os.environ.get("BACKUP_CKPT")          # e.g. /content/drive/MyDrive/MyVoice/resume.ckpt
FINAL = os.environ.get("FINAL_CKPT", "final.ckpt")
EVERY = int(os.environ.get("BACKUP_EVERY", "100"))
INIT = os.environ.get("INIT_WEIGHTS")          # base voice checkpoint to start from (fresh run only)


def _atomic_save(trainer, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    trainer.save_checkpoint(str(tmp))
    os.replace(tmp, path)


class WarmStart(Callback):
    """Copy the base voice's weights into the model before training (fine-tuning)."""

    def setup(self, trainer, pl_module, stage):
        if not INIT or getattr(self, "_done", False):
            return
        import torch

        state = torch.load(INIT, map_location="cpu", weights_only=False)["state_dict"]
        own = pl_module.state_dict()
        ok = {k: v for k, v in state.items() if k in own and own[k].shape == v.shape}
        pl_module.load_state_dict(ok, strict=False)
        print(f"Warm start from {Path(INIT).name}: loaded {len(ok)}/{len(own)} tensors", flush=True)
        self._done = True


class Progress(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        left = (trainer.max_epochs or 0) - epoch
        m = trainer.callback_metrics
        if epoch % 10 == 0:
            print(f"epoch {epoch}  ({left} to go)  gen={float(m.get('loss_g', 0)):.2f}  disc={float(m.get('loss_d', 0)):.2f}", flush=True)
        if BACKUP and epoch % EVERY == 0:
            _atomic_save(trainer, BACKUP)
            print(f"  saved resume point -> {BACKUP}", flush=True)

    def on_train_end(self, trainer, pl_module):
        _atomic_save(trainer, FINAL)
        print(f"Training finished. Final checkpoint: {FINAL}", flush=True)
        if BACKUP:
            _atomic_save(trainer, BACKUP)


if __name__ == "__main__":
    import logging
    import torch

    logging.basicConfig(level=logging.WARNING)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    VitsLightningCLI(
        VitsModel,
        VitsDataModule,
        save_config_callback=None,
        trainer_defaults={
            "max_epochs": -1,
            "callbacks": [WarmStart(), Progress()],
            "enable_checkpointing": False,
            "logger": False,
            "enable_progress_bar": False,
        },
    )


In [ ]:
%%writefile /content/export_voice.py
"""Export a Piper checkpoint to ONNX (legacy TorchScript exporter, opset 15) for phone playback."""
import sys, inspect, torch
from piper.train.vits.lightning import VitsModel

ckpt, out = sys.argv[1], sys.argv[2]
torch.manual_seed(1234)
model = VitsModel.load_from_checkpoint(ckpt, map_location="cpu")
g = model.model_g
g.eval()
with torch.no_grad():
    g.dec.remove_weight_norm()

def infer_forward(text, text_lengths, scales, sid=None):
    return g.infer(text, text_lengths, noise_scale=scales[0], length_scale=scales[1],
                   noise_scale_w=scales[2], sid=sid)[0].unsqueeze(1)
g.forward = infer_forward

seq = torch.randint(0, g.n_vocab, (1, 50), dtype=torch.long)
lens = torch.LongTensor([50])
scales = torch.FloatTensor([0.667, 1.0, 0.8])
sid = torch.LongTensor([0]) if g.n_speakers > 1 else None
kw = {}
if "dynamo" in inspect.signature(torch.onnx.export).parameters:
    kw["dynamo"] = False
torch.onnx.export(g, (seq, lens, scales, sid), out, opset_version=15,
                  input_names=["input", "input_lengths", "scales", "sid"], output_names=["output"],
                  dynamic_axes={"input": {0: "batch_size", 1: "phonemes"}, "input_lengths": {0: "batch_size"},
                                "output": {0: "batch_size", 2: "time"}}, **kw)
print("Exported", out)


In [ ]:
#@title 4 · Unpack your recordings
import zipfile, shutil, csv, wave, contextlib
DATA = '/content/data'
shutil.rmtree(DATA, ignore_errors=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall(DATA)
rows = [l.strip().split('|', 1) for l in open(f'{DATA}/metadata.csv', encoding='utf-8') if '|' in l]
secs = 0.0
for name, _ in rows:
    with contextlib.closing(wave.open(f'{DATA}/wavs/{name}')) as w:
        secs += w.getnframes() / w.getframerate()
print(f'{len(rows)} recordings, {secs/60:.1f} minutes of speech')
if secs < 240:
    print('⚠️ Under 4 minutes — it will train, but more recordings give a much closer voice.')


In [ ]:
#@title 5 · Download the starting voice
from huggingface_hub import list_repo_files, hf_hub_download
RESUME = f'{WORK}/resume.ckpt'
if START_OVER and os.path.exists(RESUME):
    os.remove(RESUME)
BASE = None
if not os.path.exists(RESUME):
    folder = f'en/en_US/{BASE_VOICE}/medium/'
    ckpts = [f for f in list_repo_files('rhasspy/piper-checkpoints', repo_type='dataset') if f.startswith(folder) and f.endswith('.ckpt')]
    assert ckpts, f'No checkpoint found for {BASE_VOICE}'
    BASE = hf_hub_download('rhasspy/piper-checkpoints', ckpts[0], repo_type='dataset', local_dir='/content/base')
    print('Starting voice:', ckpts[0])
else:
    print('Found an unfinished run on Drive — continuing it.')


In [ ]:
#@title 6 · Train (1–3 hours — keep this tab open)
env = dict(os.environ, BACKUP_CKPT=RESUME, FINAL_CKPT='/content/final.ckpt', BACKUP_EVERY='100')
if BASE:
    env['INIT_WEIGHTS'] = BASE
cmd = [sys.executable, '/content/train_voice.py', 'fit',
       '--data.voice_name', VOICE_NAME,
       '--data.csv_path', f'{DATA}/metadata.csv',
       '--data.audio_dir', f'{DATA}/wavs',
       '--model.sample_rate', '22050',
       '--data.espeak_voice', 'en-us',
       '--data.cache_dir', '/content/cache',
       '--data.config_path', f'/content/{VOICE_NAME}.onnx.json',
       '--data.batch_size', str(BATCH_SIZE),
       '--data.validation_split', '0',
       '--data.num_test_examples', '0',
       '--trainer.max_epochs', str(EPOCHS),
       '--trainer.accelerator', 'gpu', '--trainer.devices', '1',
       '--trainer.precision', '32']
if not BASE:
    cmd += ['--ckpt_path', RESUME]
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if any(k in line for k in ('epoch ', 'saved', 'Warm start', 'finished', 'Error', 'error', 'Traceback', 'File "')):
        print(line, end='')
assert p.wait() == 0, 'Training stopped with an error (see above).'


In [ ]:
#@title 7 · Export for the phone and listen
import shutil, wave
from IPython.display import Audio, display
onnx_path = f'/content/{VOICE_NAME}.onnx'
subprocess.run([sys.executable, '/content/export_voice.py', '/content/final.ckpt', onnx_path], check=True)
from piper import PiperVoice
voice = PiperVoice.load(onnx_path)
with wave.open('/content/sample.wav', 'wb') as w:
    voice.synthesize_wav('Hi everyone, welcome back. Today I am testing my own voice model, trained on my recordings.', w)
display(Audio('/content/sample.wav'))
shutil.copy(onnx_path, f'{WORK}/{VOICE_NAME}.onnx')
shutil.copy(f'/content/{VOICE_NAME}.onnx.json', f'{WORK}/{VOICE_NAME}.onnx.json')
if os.path.exists(RESUME):
    os.remove(RESUME)  # run finished; free 850 MB of Drive space
print(f'✅ Saved to Google Drive → MyVoice/{VOICE_NAME}.onnx and {VOICE_NAME}.onnx.json')
print('On your phone: download both from Drive, then My Voice → Train → Import my model.')
print('Want it even closer? Record more sentences in the app, export again, upload, and Run all.')
